In [2]:
import joblib
import pandas as pd
from datetime import datetime

cong_model = joblib.load("cv2.pkl")
time_model = joblib.load("90.pkl")

# ---- user input ----
distance  = 15.77
time_hr   = 13.3
weather   = "Fog"
road_type = "urban"
date_str  = "2024-12-14"   # user picks this date from frontend

# ---- extract date features automatically ----
date        = datetime.strptime(date_str, "%Y-%m-%d")
month       = date.month           # 10
day_of_week = date.strftime("%A")  # "Monday"

# ---- build base input frames (all zeros matching model's expected columns) ----
X_cong = pd.DataFrame([[0] * len(cong_model.feature_names_in_)],
                       columns=cong_model.feature_names_in_)

X_time = pd.DataFrame([[0] * len(time_model.feature_names_in_)],
                       columns=time_model.feature_names_in_)

# ---- fill features for both models ----
for df in [X_cong, X_time]:
    df["distance_km"]   = distance
    df["time_in_hours"] = time_hr
    df["month"]         = month        # derived from date

    # road_type one-hot (baseline = congested, so only set if not congested)
    if road_type == "urban":
        df["road_type_urban"]   = 1
    elif road_type == "freeway":
        df["road_type_freeway"] = 1
    # if road_type == "congested" → all road_type columns stay 0 (baseline)

    # day_of_week one-hot (baseline = Friday, so only set if not Friday)
    col = f"day_of_week_{day_of_week}"
    if col in df.columns:
        df[col] = 1
    # if day_of_week == "Friday" → all day_of_week columns stay 0 (baseline)

    # weather one-hot (baseline = Clear, so only set if not Clear)
    col = f"weather_{weather}"
    if col in df.columns:
        df[col] = 1
    # if weather == "Clear" → all weather columns stay 0 (baseline)

# ---- step 1: predict congestion index ----
pred_cong = cong_model.predict(X_cong)[0]

# ---- step 2: feed congestion into travel time model ----
X_time["congestion_index"] = pred_cong

# ---- step 3: predict travel time ----
pred_time = time_model.predict(X_time)[0]

print(f"Date:             {date_str} ({day_of_week})")
print(f"Month:            {month}")
print(f"Distance:         {distance} km")
print(f"Time:             {time_hr} hrs")
print(f"Weather:          {weather}")
print(f"Road Type:        {road_type}")
print(f"---")
print(f"Predicted Congestion Index: {pred_cong:.3f}")
print(f"Predicted Travel Time:      {pred_time:.1f} minutes")

Date:             2024-12-14 (Saturday)
Month:            12
Distance:         15.77 km
Time:             13.3 hrs
Weather:          Fog
Road Type:        urban
---
Predicted Congestion Index: 0.257
Predicted Travel Time:      30.9 minutes


In [ ]:
joblib.dump(cong_model, "congestion_prediction_model.pkl")
joblib.dump(time_model, "traffic_congestion_model.pkl")